# COVID × AD comorbidity sub-maps

The same construction as `4_10`, but with the **AD BEL knowledge graph downstream** instead of the PD disease maps.

The AD KG has no CellDesigner representation, so its side is derived from its **influence-graph projection** — the one `2_05` defines and documents — and built as an in-memory `CellDesignerMap` by `commute_dm.bel_submaps`, which is then merged into the same `source_map` as the stored COVID activity-flow elements. After that merge nothing distinguishes the two sides: a BEL node id resolves to a species of `source_map.model` exactly as a stored one does.

Nothing here writes to the database.

In [1]:
%store -r

In [2]:
import commute_dm.bel_submaps
import commute_dm.core
import commute_dm.submaps
import commute_dm.utils
import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
def make_backend():
    return momapy_kb.lpg.backends.neo4j.Neo4jBackend(
        hostname=credentials.NEO4J_URI,
        username=credentials.NEO4J_USERNAME,
        password=credentials.NEO4J_PASSWORD,
        notifications_min_severity="off",
    )

`MAX_LEVELS` is much smaller than `4_10`'s `[2, 3, 4, 5, 6]`. The AD influence graph is far denser than an activity-flow map: from the interface seeds the median downstream selection is 3 nodes at one hop, 6 at two and 18 at three, but 140 at four and about 960 unbounded — past three hops the maps stop being readable.

In [4]:
MAX_LEVELS = [1, 2, 3]
MIN_N_NODES = 5
UPSTREAM_COLLECTION_NAME = "COVID_DM_CD_AF"
DOWNSTREAM_COLLECTION_NAME = "AD_KG_BEL"
INTERFACE_COLLECTION_NAMES = (
    UPSTREAM_COLLECTION_NAME,
    DOWNSTREAM_COLLECTION_NAME,
)

We compute the interface between the COVID activity-flow maps and the AD KG, and load what the sub-maps are made of.

`load_submap_inputs` hydrates every stored map of the **COVID** collection (about a minute — half of what `4_10` pays, since only one CellDesigner collection is loaded), then builds the AD projection as momapy objects (~0.4 s) and merges the two. It also returns the **seed expansion**: the UniProt annotations that define the interface sit on the KG's `p(HGNC:X)` nodes, but BEL keeps a protein's causal wiring on its activity form `act(p(HGNC:X))`, so the activity forms are added to the downstream seeds.

In [5]:
with momapy_kb.lpg.session.Session(make_backend()) as session:
    interface = commute_dm.core.get_interface(session, INTERFACE_COLLECTION_NAMES)
    (
        influences,
        source_map,
        node_id_to_object,
        seed_expansion,
        bel_stats,
    ) = commute_dm.core.load_submap_inputs(
        session, UPSTREAM_COLLECTION_NAME, DOWNSTREAM_COLLECTION_NAME
    )
(
    len(interface),
    len(source_map.model.species),
    len(source_map.model.modulations),
    len(seed_expansion),
)

(189, 6148, 8417, 79)

## What the BEL side became

See `commute_dm.bel_terms` for the four identity invariants these numbers are the evidence for.

In [6]:
# What the AD knowledge graph became. `n_species_collapsed` is the projected
# node ids that describe identically once `ma()` is dropped (activities) or
# once whitespace in a `var()` is normalised; the walk is keyed on node ids,
# so only the drawing merges. `n_templates_interned_from_source_map` is the
# number of BEL templates that would otherwise have been a dangling
# `<proteinReference>`.
bel_stats

{'n_terms_described': 4513,
 'n_templates': 1738,
 'n_templates_interned_from_source_map': 140,
 'n_compartments': 4,
 'n_compartment_roots': 1,
 'n_projected_node_ids': 3979,
 'n_species': 3858,
 'n_species_collapsed': 121,
 'n_subunits': 1782,
 'max_complex_depth': 2,
 'n_modifications': 227,
 'n_structural_states': 194,
 'n_active': 471,
 'n_named_by_suffix': 11,
 'n_drawn_compartments': 4,
 'n_self_loop_edges_dropped': 0,
 'n_modulations': 4729,
 'n_mapping_entries': 6065}

In [7]:
commute_dm.utils.remake_dir(INTERFACE_AD_ANALYSIS_GRAPHS_DIR)

We assemble and write the sub-maps upstream of the COVID seeds and downstream of the AD seeds, joined by a synthetic central node standing for the interface protein.

Species are coloured by which walk reached them: blue upstream (stored COVID activity-flow elements, drawn inside their compartment boxes), green downstream (AD BEL nodes, which carry no compartment and so sit outside every box), red for the central node. A BEL species is named by its full BEL string, so the glyph cannot show that `act(p(X))` is an activity but the label can.

In [8]:
with momapy_kb.lpg.session.Session(make_backend()) as session:
    stats_df = commute_dm.core.make_and_write_submaps_from_interface(
        session=session,
        interface=interface,
        influences=influences,
        source_map=source_map,
        node_id_to_object=node_id_to_object,
        output_dir_path=INTERFACE_AD_ANALYSIS_GRAPHS_DIR,
        upstream_collection_name=UPSTREAM_COLLECTION_NAME,
        downstream_collection_name=DOWNSTREAM_COLLECTION_NAME,
        max_levels=MAX_LEVELS,
        min_n_nodes=MIN_N_NODES,
        downstream_node_id_expansion=seed_expansion,
    )
stats_df

,identifier,display_name,max_level,n_species,n_subunits,n_modifications,n_modulations,n_gates,n_compartments,n_templates,n_layout_elements
0,P49662,CASP4,2,25,2,0,36,0,3,12,63
1,P49662,CASP4,3,190,13,5,326,0,6,118,521
2,P01375,TNF,1,58,0,0,68,0,6,35,131
3,P01375,TNF,2,113,12,2,222,0,6,70,340
4,P01375,TNF,3,499,56,31,1084,1,9,286,1594
...,...,...,...,...,...,...,...,...,...,...,...
96,P01031,C5,3,427,46,28,946,0,6,251,1377
97,Q96P20,NLRP3,2,75,18,1,146,3,6,41,246
98,Q96P20,NLRP3,3,480,62,29,1033,3,12,272,1543
99,P06702,S100A9,2,221,9,6,414,0,5,129,638


How many maps came out per level. `MIN_N_NODES` applies to **both** directions, and it is the COVID upstream side that usually falls short at one hop — not the AD side.

In [9]:
stats_df.groupby("max_level")["identifier"].count()

max_level
1    16
2    40
3    45
Name: identifier, dtype: int64

## Read-back

Every written file is read back. This is the assertion the identity invariants exist for: a violation writes a valid-looking file and fails here with a `KeyError`, so a regression must be loud rather than discovered in CellDesigner.

In [10]:
import glob
import os.path

import momapy.io.core

written_file_paths = sorted(
    glob.glob(os.path.join(INTERFACE_AD_ANALYSIS_GRAPHS_DIR, "*", "*.xml"))
)
read_back_failures = []
for written_file_path in written_file_paths:
    try:
        read_back = momapy.io.core.read(written_file_path, reader="celldesigner").obj
        assert read_back.model.species
    except Exception as exception:
        read_back_failures.append((written_file_path, repr(exception)))
assert not read_back_failures, read_back_failures[:3]
len(written_file_paths)

101